# Resultants

*An expository note, with everything checked in Sage.*

Given $f,g \in K[x]$ over a field, we can ask whether they have a common root without
factoring either of them: the **resultant** $\operatorname{Res}(f,g)$ is a single element of $K$,
a polynomial expression in the coefficients, that vanishes exactly when they do have one.
Being polynomial in the coefficients is the whole point — it means the answer varies
*algebraically* with $f$ and $g$, so it survives specialisation: plugging in values for
parameters, or reducing coefficients modulo a prime.

The target of the note is this theorem, which is the reduction-mod-$p$ instance:

> **Theorem 5.** Let $f,g \in \mathbb{Z}[x]$ have degrees $m,n \ge 1$ and leading coefficients
> $a_m, b_n$, and let $p$ be prime. Then
> $$p \mid \operatorname{Res}(f,g) \iff \bar f, \bar g \in \mathbb{F}_p[x] \text{ have a common factor of positive degree, or } p \mid a_m \text{ and } p \mid b_n.$$
> In particular, if $p \nmid a_m$ or $p \nmid b_n$ (e.g. if either polynomial is monic):
> $$p \mid \operatorname{Res}(f,g) \iff \bar f \text{ and } \bar g \text{ have a common root in } \overline{\mathbb{F}_p}.$$

Two warnings about the statement, both of which the note will make precise.

* The gcd is taken **after** reducing mod $p$. It is *not* true that $p \mid \operatorname{Res}(f,g)$
  iff $p$ divides some gcd computed in $\mathbb{Z}[x]$: for $f = x$ and $g = x+p$ we have
  $\gcd(f,g) = 1$ in $\mathbb{Z}[x]$ while $\operatorname{Res}(f,g) = p$. What happens is that
  $f$ and $g$ are coprime over $\mathbb{Q}$ but *become* non-coprime mod $p$, and the resultant
  is exactly the record of all such primes.
* The leading-coefficient clause is a genuine exception, not pedantry — Example 5.3 exhibits it.

The route: Sylvester matrix (§1), what it means as a linear map (§2), the Bézout identity (§3),
the vanishing criterion over a field (§4), the product formula (§5), base change (§6), and then
Theorem 5 falls out in three lines (§7). §8 has applications.

## 1. The Sylvester matrix

Fix **formal degrees** $m, n \ge 1$ and write
$$f = a_mx^m + \dots + a_0, \qquad g = b_nx^n + \dots + b_0,$$
allowing $a_m$ or $b_n$ to be $0$. Insisting on formal rather than actual degrees looks like
fussiness, but it is precisely what makes the theory behave under specialisation; keep an eye on it.

The **Sylvester matrix** $\operatorname{Syl}_{m,n}(f,g)$ is the $(m+n)\times(m+n)$ matrix with
$n$ rows of shifted $f$-coefficients on top of $m$ rows of shifted $g$-coefficients:
$$\operatorname{Syl}_{m,n}(f,g) = \begin{pmatrix}
a_m & a_{m-1} & \cdots & a_0 & & \\
 & a_m & a_{m-1} & \cdots & a_0 & \\
 & & \ddots & & & \ddots \\
b_n & b_{n-1} & \cdots & b_0 & & \\
 & b_n & b_{n-1} & \cdots & b_0 & \\
 & & \ddots & & & \ddots
\end{pmatrix}$$
and the **resultant** is
$$\operatorname{Res}_{m,n}(f,g) := \det \operatorname{Syl}_{m,n}(f,g) \in R,$$
a polynomial with integer coefficients in the $a_i, b_j$. When $m = \deg f$ and $n = \deg g$ we
just write $\operatorname{Res}(f,g)$.

In [1]:
R.<x> = ZZ[]

def syl(f, g, m, n):
    "Sylvester matrix of f, g with respect to the FORMAL degrees m, n."
    a = f.list() + [0]*(m+1-len(f.list())); a.reverse()    # a[0] is the coeff of x^m
    b = g.list() + [0]*(n+1-len(g.list())); b.reverse()
    rows =  [[0]*i + a + [0]*(n-1-i) for i in range(n)]    # n shifted rows of f
    rows += [[0]*j + b + [0]*(m-1-j) for j in range(m)]    # m shifted rows of g
    return matrix(rows)

f = x^2 - 1
g = 3*x^2 + x + 1
show(syl(f, g, 2, 2))
print("det          =", syl(f, g, 2, 2).det())
print("Sage's Res   =", f.resultant(g))
print("Sage's Syl   =", syl(f,g,2,2) == f.sylvester_matrix(g))

[ 1  0 -1  0]
[ 0  1  0 -1]
[ 3  1  1  0]
[ 0  3  1  1]

det          = 15
Sage's Res   = 15
Sage's Syl   = True


## 2. What the matrix is doing

The definition looks unmotivated until you read the matrix as a **linear map**. Consider
$$\Phi : K[x]_{<n} \times K[x]_{<m} \longrightarrow K[x]_{<m+n}, \qquad (u,v) \longmapsto uf + vg,$$
where $K[x]_{<d}$ denotes polynomials of degree $< d$, a $d$-dimensional space. Source and target
both have dimension $m+n$, so $\Phi$ is a square matrix, and reading it off in the monomial bases
$(x^{n-1},\dots,1)$, $(x^{m-1},\dots,1)$ and $(x^{m+n-1},\dots,1)$ gives exactly
$\operatorname{Syl}_{m,n}(f,g)$: the $i$-th row is the coefficient vector of $x^{n-1-i}f$, the
$(n+j)$-th that of $x^{m-1-j}g$.

$$\boxed{\;\operatorname{Res}_{m,n}(f,g) = \det \Phi\;}$$

So the resultant measures the failure of $\Phi$ to be invertible, i.e. whether **the Bézout
equation $uf + vg = 1$ can be solved with the degree bounds $\deg u < n$, $\deg v < m$.**
Everything below is a consequence of this one observation. Note also, from the boxed formula,
the two facts one would want:
* $\operatorname{Res}$ is a polynomial in the coefficients with coefficients in $\mathbb{Z}$;
* it does not care which ring the coefficients live in.

## 3. The Bézout identity: $\operatorname{Res}(f,g)$ lies in the ideal $(f,g)$

**Proposition 1.** *Over any commutative ring $R$ there are $u,v \in R[x]$ with $\deg u < n$,
$\deg v < m$ and*
$$u f + v g = \operatorname{Res}_{m,n}(f,g).$$

*Proof.* Write $S = \operatorname{Syl}_{m,n}(f,g)$ and let $\mathbf{x} = (x^{m+n-1},\dots,x,1)^{T}$.
By construction
$$S\,\mathbf{x} = (x^{n-1}f,\; \dots,\; f,\; x^{m-1}g,\; \dots,\; g)^{T}.$$
Multiply on the left by the adjugate and use $\operatorname{adj}(S)\,S = \det(S)\,\mathrm{I}$:
$$\det(S)\,\mathbf{x} = \operatorname{adj}(S)\cdot(x^{n-1}f,\dots,f,x^{m-1}g,\dots,g)^{T}.$$
The last coordinate of $\mathbf{x}$ is $1$, so reading off the last row gives
$\det(S) = uf + vg$, where $u$ and $v$ have as coefficients the entries of the last row of
$\operatorname{adj}(S)$ — integer polynomials in the $a_i,b_j$ — and the degree bounds are visible
from which entries multiply which $x^kf$ or $x^kg$. $\blacksquare$

Two immediate consequences, both used later:

**Corollary 1.1.** *Any common divisor $h$ of $f$ and $g$ in $R[x]$ divides the constant
$\operatorname{Res}_{m,n}(f,g)$. Hence over a field, if $f,g$ have a common factor of positive
degree, then $\operatorname{Res}_{m,n}(f,g) = 0.$*

**Corollary 1.2.** *If $f,g \in \mathbb{Z}[x]$ are coprime in $\mathbb{Q}[x]$, then the ideal
$(f,g) \subseteq \mathbb{Z}[x]$ contains the nonzero integer $\operatorname{Res}(f,g)$.* This is the
standard way of producing an explicit integer in an elimination ideal.

In [2]:
# Proposition 1, explicitly, via the adjugate
f = x^3 + 2*x + 5
g = x^2 - 3
m, n = f.degree(), g.degree()

S   = syl(f, g, m, n)
row = S.adjugate().row(m+n-1)                 # last row of adj(S)
u   = sum(row[i]     * x^(n-1-i) for i in range(n))
v   = sum(row[n+j]   * x^(m-1-j) for j in range(m))

print("u =", u, "   (deg < n =", n, ")")
print("v =", v, "   (deg < m =", m, ")")
print("u*f + v*g =", u*f + v*g)
print("Res(f,g)  =", f.resultant(g))
print("identity holds:", u*f + v*g == f.resultant(g))

u = -5*x + 5    (deg < n = 2 )
v = 5*x^2 - 5*x + 25    (deg < m = 3 )
u*f + v*g = -50
Res(f,g)  = -50
identity holds: True


## 4. The vanishing criterion over a field

**Theorem 2.** *Let $K$ be a field and $f,g \in K[x]$ with formal degrees $m,n \ge 1$, and suppose
$a_m$ and $b_n$ are not both $0$. Then*
$$\operatorname{Res}_{m,n}(f,g) = 0 \iff f,g \text{ have a common factor of positive degree in } K[x] \iff f,g \text{ have a common root in } \overline{K}.$$

*Proof.* Over a field $\det \Phi = 0$ iff $\Phi$ is not injective, i.e. iff there is a pair
$(u,v) \ne (0,0)$ with $\deg u < n$, $\deg v < m$ and
$$uf + vg = 0. \tag{$\ast$}$$

($\Leftarrow$) If $h \mid f$, $h \mid g$ with $\deg h \ge 1$, write $f = hf_1$, $g = hg_1$ and take
$(u,v) = (g_1, -f_1)$: then $uf + vg = g_1hf_1 - f_1hg_1 = 0$, and
$\deg g_1 = \deg g - \deg h < n$, $\deg f_1 < m$. (Degenerate cases: if $g = 0$ take $(u,v)=(0,1)$,
and symmetrically.) So $\Phi$ is not injective.

($\Rightarrow$) Suppose $(\ast)$ has a nontrivial solution but $\gcd(f,g) = 1$. By hypothesis
$a_m \ne 0$ or $b_n \ne 0$; say $a_m \ne 0$, so $\deg f = m$ exactly (the other case is symmetric).
If $g = 0$ then coprimality makes $f$ a nonzero constant, contradicting $\deg f = m \ge 1$; so
$g \ne 0$. From $uf = -vg$ we get $g \mid uf$, and $\gcd(f,g)=1$ gives $g \mid u$, say $u = gw$.
Substituting, $g(wf + v) = 0$, and $K[x]$ is a domain, so $v = -wf$. If $w \ne 0$ this forces
$\deg v \ge \deg f = m$, contradicting $\deg v < m$. So $w = 0$, hence $u = 0$, hence $vg = 0$,
hence $v = 0$ — contradicting $(u,v)\ne(0,0)$.

Finally, a common factor of positive degree has a root in $\overline{K}$, and conversely a common
root $\alpha$ gives the common factor $x - \alpha$ over $\overline{K}$, hence
$\gcd(f,g) \neq 1$ already over $K$ (the gcd is unchanged by field extension). $\blacksquare$

The excluded case is real, and worth isolating:

**Lemma 3.** *If $a_m = b_n = 0$ then $\operatorname{Res}_{m,n}(f,g) = 0$, whatever $f$ and $g$ are.*

*Proof.* Then $\deg(uf + vg) \le m+n-2$ for all $u,v$ in the source, so the image of $\Phi$ lies in
the proper subspace $K[x]_{<m+n-1}$ and $\det\Phi = 0$. (Equivalently: the first column of the
Sylvester matrix is zero.) $\blacksquare$

Applying Theorem 2 over $K = \mathbb{Q}$ already gives the classical statement: for
$f,g \in \mathbb{Z}[x]$ of true degrees $m,n \ge 1$,
$$\operatorname{Res}(f,g) = 0 \iff f,g \text{ have a common factor of positive degree in } \mathbb{Q}[x].$$
Theorem 5 is what you get by running the same argument over $\mathbb{F}_p$ instead.

## 5. The product formula

Not needed for Theorem 5, but it is the other classical face of the resultant and explains the
normalisation $a_m^n$.

**Proposition 4.** *Let $K$ be a field, $\deg f = m \ge 1$ exactly, and $n \ge \deg g$. Then*
$$\operatorname{Res}_{m,n}(f,g) = a_m^{\,n}\cdot \det\big(\,\cdot g : K[x]/(f) \to K[x]/(f)\,\big) = a_m^{\,n}\prod_{i=1}^{m} g(\alpha_i),$$
*where $\alpha_1,\dots,\alpha_m$ are the roots of $f$ in $\overline{K}$ with multiplicity. Symmetrically,
if also $\deg g = n$ with roots $\beta_j$,*
$$\operatorname{Res}(f,g) = a_m^{\,n} b_n^{\,m} \prod_{i,j}(\alpha_i - \beta_j) = (-1)^{mn}\operatorname{Res}(g,f).$$

*Proof.* Since $\deg f = m$, division with remainder says every $h$ with $\deg h < m+n$ is uniquely
$uf + r$ with $\deg u < n$, $\deg r < m$. So
$$\mathcal{B} = (x^{n-1}f,\;\dots,\;f,\;x^{m-1},\;\dots,\;1)$$
is a basis of $K[x]_{<m+n}$. Write $\Phi$ from §2 in the basis $\mathcal{B}$ on the target: since
$\Phi(u,0) = uf$ and $\Phi(0,v) = vg = q_vf + (vg \bmod f)$, the matrix is block upper triangular
$$\begin{pmatrix} \mathrm{I}_n & Q \\ 0 & M\end{pmatrix},$$
where $M$ is the matrix of $v \mapsto vg \bmod f$ on $K[x]_{<m} \cong K[x]/(f)$ — that is,
multiplication by $g$ on $K[x]/(f)$. Its determinant is $\det M$. Changing back from $\mathcal{B}$ to
the monomial basis costs the determinant of the change-of-basis matrix, which is block triangular
$\left(\begin{smallmatrix} A & *\\ 0 & \mathrm{I}_m\end{smallmatrix}\right)$ with $A$ upper triangular
with $a_m$ on the diagonal, hence $a_m^n$. So $\det\operatorname{Syl}_{m,n}(f,g) = a_m^n \det M$.

For the product: over $\overline{K}$, the Chinese remainder theorem gives
$\overline{K}[x]/(f) \cong \prod_i \overline{K}[x]/\big((x-\alpha_i)^{e_i}\big)$, and on each factor
multiplication by $g$ is upper triangular in the basis $1, (x-\alpha_i), \dots$ with $g(\alpha_i)$
repeated on the diagonal. So $\det M = \prod_i g(\alpha_i)^{e_i}$. $\blacksquare$

Note the formula depends on $n$ only through $a_m^n$; that is exactly the correction factor relating
formal to true degrees, $\operatorname{Res}_{m,n}(f,g) = a_m^{\,n-\deg g}\operatorname{Res}_{m,\deg g}(f,g)$.

In [3]:
f = x^3 + 2*x + 5
g = x^2 - 3
m, n = f.degree(), g.degree()

# determinant of multiplication by g on Q[x]/(f)
RQ.<y> = QQ[]
A  = RQ.quotient(RQ(f))
Mg = matrix([ (A(RQ(g)) * A(y^i)).lift().padded_list(m) for i in range(m) ])
print("lc(f)^n * det(mult by g) =", f.leading_coefficient()^n * Mg.det())

# product over the roots of f
prod_roots = QQ(f.leading_coefficient()^n * prod(g(a) for a in f.roots(QQbar, multiplicities=False)))
print("lc(f)^n * prod g(alpha_i) =", prod_roots)
print("Res(f,g)                  =", f.resultant(g))
print("symmetry (-1)^(mn) Res(g,f) =", (-1)^(m*n) * g.resultant(f))

lc(f)^n * det(mult by g) = -50
lc(f)^n * prod g(alpha_i) = -50
Res(f,g)                  = -50
symmetry (-1)^(mn) Res(g,f) = -50


## 6. Base change — the reason resultants are useful

**Lemma 5 (specialisation).** *Let $\varphi : R \to S$ be a ring homomorphism, extended to
$R[x]\to S[x]$ coefficientwise. Then, for the **same** formal degrees $m,n$,*
$$\operatorname{Res}_{m,n}\big(\varphi f, \varphi g\big) = \varphi\big(\operatorname{Res}_{m,n}(f,g)\big).$$

*Proof.* The entries of $\operatorname{Syl}_{m,n}(f,g)$ are coefficients of $f$ and $g$, so
$\operatorname{Syl}_{m,n}(\varphi f,\varphi g) = \varphi\big(\operatorname{Syl}_{m,n}(f,g)\big)$
entrywise; and $\det$ is a polynomial with integer coefficients in the entries, which every ring
homomorphism respects. $\blacksquare$

That is the entire content, and it is trivial — but only because we fixed the formal degrees. This
is where the bookkeeping of §1 pays off: if $\varphi$ kills the leading coefficient of $g$, then
$\operatorname{Res}_{m,n}(\varphi f, \varphi g)$ is *not* the resultant of $\varphi f, \varphi g$ in
their own degrees; by the last remark of §5 the two differ by $\varphi(a_m)^{\,n - \deg \varphi g}$.
The lemma stays true regardless, since it never mentions true degrees.

## 7. Theorem 5

**Theorem 5.** *Let $f,g \in \mathbb{Z}[x]$ have true degrees $m,n \ge 1$ and leading coefficients
$a_m,b_n$, and let $p$ be a prime. Write $\bar{\;\cdot\;}$ for reduction mod $p$. Then*
$$p \mid \operatorname{Res}(f,g) \iff \Big(\gcd(\bar f,\bar g) \ne 1 \text{ in } \mathbb{F}_p[x]\Big) \ \text{ or } \ \Big(p \mid a_m \text{ and } p \mid b_n\Big).$$

*Proof.* Reduction $\varphi : \mathbb{Z} \to \mathbb{F}_p$ is a ring homomorphism, so Lemma 5 with
the formal degrees $m,n$ gives
$$\operatorname{Res}(f,g) \bmod p \;=\; \operatorname{Res}_{m,n}(\bar f, \bar g).$$
Hence $p \mid \operatorname{Res}(f,g)$ iff $\operatorname{Res}_{m,n}(\bar f,\bar g) = 0$ in $\mathbb{F}_p$.
Now apply §4 over the field $K = \mathbb{F}_p$, with formal degrees $m,n$:

* if $\bar a_m = \bar b_n = 0$, Lemma 3 gives $\operatorname{Res}_{m,n}(\bar f,\bar g)=0$;
* otherwise Theorem 2 applies and $\operatorname{Res}_{m,n}(\bar f,\bar g) = 0$ iff $\bar f,\bar g$
  have a common factor of positive degree in $\mathbb{F}_p[x]$, equivalently a common root in
  $\overline{\mathbb{F}_p}$. $\blacksquare$

So the proof is: *base change is free if you track formal degrees, and over a field the resultant
vanishes iff there is a common factor.* All the work sits in Theorem 2, which is the linear algebra
of §2.

**Remark 5.1 (why "gcd of $f$ and $g$" must mean the gcd mod $p$).** The gcd in $\mathbb{Z}[x]$ is
the wrong object: $f = x$ and $g = x+p$ have $\gcd(f,g) = 1$ in $\mathbb{Z}[x]$, yet
$\operatorname{Res}(f,g) = p$. The gcd here is legitimate even though $\mathbb{Z}[x]$ is not a
principal ideal domain: by Gauss's lemma $\mathbb{Z}[x]$ is a *UFD*, and in a UFD gcds exist and are
unique up to units — $x$ and $x+p$ are non-associate irreducibles, so their only common divisors are
$\pm 1$. The point of the theorem is that the resultant sees the primes at which two globally coprime
polynomials *become* non-coprime.

**Remark 5.1a (divisibility versus ideals).** It is worth being careful about two notions that
coincide over a field but come apart over $\mathbb{Z}$:

| | $f = x,\ g = x+p$ |
|---|---|
| divisibility | $\gcd(f,g) = 1$ |
| ideals | $(f,g) = (x,p)$, a maximal ideal, $\mathbb{Z}[x]/(x,p)\cong\mathbb{F}_p$, so $1 \notin (f,g)$ |

$\mathbb{Z}[x]$ is a UFD but not a Bézout ring, so **the gcd does not generate the ideal**. The
resultant lives on the *ideal* side of this distinction: by Proposition 1,
$\operatorname{Res}(f,g) \in (f,g)\cap\mathbb{Z}$, and here $(x,x+p)\cap\mathbb{Z} = p\mathbb{Z}$ — which
is exactly why $\operatorname{Res} = p$. So Theorem 5 can be restated ideal-theoretically: away from
the leading-coefficient case,
$$p \mid \operatorname{Res}(f,g) \iff (f,g,p) \ne \mathbb{Z}[x] \iff V(f,g) \subset \operatorname{Spec}\mathbb{Z}[x] \text{ has a point above } p .$$
Reducing mod $p$ replaces $\mathbb{Z}[x]$ by the PID $\mathbb{F}_p[x]$, where "gcd $\ne 1$" and "the
ideal is proper" agree again — which is what makes a gcd formulation of the criterion possible at all.

Note finally that $\operatorname{Res}(f,g)$ lies in $(f,g)\cap\mathbb{Z}$ but need not generate it: for
$f = x^2$, $g = x^2+p$ the contraction is $(p)$ while $\operatorname{Res}(f,g) = p^2$. The resultant
records the intersection with multiplicity.

**Remark 5.2.** Combining with §4 over $\mathbb{Q}$: if $f,g$ are coprime in $\mathbb{Q}[x]$ then
$\operatorname{Res}(f,g) \ne 0$, so only finitely many primes are "bad", and they are exactly the
prime divisors of $\operatorname{Res}(f,g)$ (together with those dividing both leading coefficients,
which divide the resultant anyway by Lemma 3).

In [4]:
# The worked example.  Res factors; each prime factor must show up as a common factor mod p.
f = x^3 + 2*x + 5
g = x^2 - 3
r = f.resultant(g)
print("Res(f,g) =", r, "=", factor(r), "\n")

for p in prime_range(30):
    Fp = GF(p); S.<t> = Fp[]
    d = gcd(S(f.list()), S(g.list()))
    print(f"p = {p:2d}   p | Res : {str(r % p == 0):5s}   gcd(fbar,gbar) = {d}")

Res(f,g) = -50 = -1 * 2 * 5^2 

p =  2   p | Res : True    gcd(fbar,gbar) = t + 1
p =  3   p | Res : False   gcd(fbar,gbar) = 1
p =  5   p | Res : True    gcd(fbar,gbar) = t^2 + 2
p =  7   p | Res : False   gcd(fbar,gbar) = 1
p = 11   p | Res : False   gcd(fbar,gbar) = 1
p = 13   p | Res : False   gcd(fbar,gbar) = 1
p = 17   p | Res : False   gcd(fbar,gbar) = 1
p = 19   p | Res : False   gcd(fbar,gbar) = 1
p = 23   p | Res : False   gcd(fbar,gbar) = 1
p = 29   p | Res : False   gcd(fbar,gbar) = 1


In [5]:
# Remark 5.1: the naive statement is false
f, g = x, x + 7
print("Res(x, x+7)  =", f.resultant(g))
print("gcd in ZZ[x] =", gcd(f, g), "    (ZZ[x] is a UFD:", R.is_unique_factorization_domain(),
      ", a PID:", R in PrincipalIdealDomains(), ")")
print("gcd mod 7    =", gcd(GF(7)['t'](f.list()), GF(7)['t'](g.list())))

# Remark 5.1a: the gcd is 1, but the ideal is the maximal ideal (x,7) -- it does not contain 1.
# (ideal arithmetic over ZZ goes through the multivariate constructor)
M = PolynomialRing(ZZ, 'x', 1)
X = M.gen(0)
for a, b in [(X, X + 7), (X^2, X^2 + 7)]:
    I = M.ideal(a, b)
    res = R(str(a)).resultant(R(str(b)))
    print(f"\n({a}, {b}):  Groebner basis over ZZ = {list(I.groebner_basis())}")
    print(f"   1 in the ideal? {M.one() in I}     7 in the ideal? {M(7) in I}"
          f"   ->  (f,g) cap ZZ = (7)")
    print(f"   Res = {res}   generates (f,g) cap ZZ: {res == 7}")

Res(x, x+7)  = 7
gcd in ZZ[x] = 1     (ZZ[x] is a UFD: True , a PID: False )
gcd mod 7    = t

(x, x + 7):  Groebner basis over ZZ = [x, 7]
   1 in the ideal? False     7 in the ideal? True   ->  (f,g) cap ZZ = (7)
   Res = 7   generates (f,g) cap ZZ: True

(x^2, x^2 + 7):  Groebner basis over ZZ = [x^2, 7]
   1 in the ideal? False     7 in the ideal? True   ->  (f,g) cap ZZ = (7)
   Res = 49   generates (f,g) cap ZZ: False


**Example 5.3 (the leading-coefficient clause is not vacuous).** Take $f = 2x^2+1$, $g = 2x^2+x+1$,
so $\operatorname{Res}(f,g) = 2$. Modulo $2$ we get $\bar f = 1$ and $\bar g = x+1$, which are
*coprime* — yet $2 \mid \operatorname{Res}(f,g)$. There is no contradiction: $2$ divides both leading
coefficients, so we are in the second clause, and Lemma 3 explains the vanishing. Concretely, the
formal-degree Sylvester matrix mod $2$ has a zero first column while the "honest" resultant of
$\bar f$ and $\bar g$ in their true degrees is not even defined ($\bar f$ is a unit).

Geometrically: the common root has escaped to infinity. This is exactly the case where the projective
plane curves $\{f = 0\}$ and $\{g = 0\}$ meet at the point at infinity after reduction.

In [6]:
f = 2*x^2 + 1
g = 2*x^2 + x + 1
S.<t> = GF(2)[]
print("Res(f,g)          =", f.resultant(g))
print("fbar, gbar mod 2  =", S(f.list()), ",", S(g.list()))
print("gcd(fbar,gbar)    =", gcd(S(f.list()), S(g.list())), "  <- coprime, yet 2 | Res")
print("both leading coefficients even:", f.leading_coefficient() % 2 == 0, g.leading_coefficient() % 2 == 0)
print("\nformal-degree Sylvester mod 2:")
show(syl(f, g, 2, 2).change_ring(GF(2)))
print("det =", syl(f, g, 2, 2).change_ring(GF(2)).det())

Res(f,g)          = 2
fbar, gbar mod 2  = 1 , t + 1
gcd(fbar,gbar)    = 1   <- coprime, yet 2 | Res
both leading coefficients even: True True

formal-degree Sylvester mod 2:


[0 0 1 0]
[0 0 0 1]
[0 1 1 0]
[0 0 1 1]

det = 0


**Remark 5.4 (gcd, not lcm).** Since $\deg\gcd + \deg\operatorname{lcm} = \deg\bar f + \deg\bar g$,
one is tempted to restate the criterion as $\deg \operatorname{lcm}(\bar f,\bar g) < m+n$. That is
equivalent to the gcd version only as long as no leading coefficient dies mod $p$, and it fails
sooner than the gcd version does: it also fires when just *one* of them dies. For $f = x$,
$g = 2x+1$ and $p = 2$ we have $\bar g = 1$, so $\deg\operatorname{lcm}(\bar f,\bar g) = 1 < m+n = 2$,
while $\operatorname{Res}(f,g) = 1$ is odd. The gcd form gets this right: $\gcd(\bar f,\bar g) = 1$,
and $2$ divides $b_n = 2$ but not $a_m = 1$, so neither clause of Theorem 5 applies.

**Remark 5.5 (the gcd statement that *is* about $\mathbb{Z}$).** There is a true statement about
honest integer gcds, immediate from Proposition 1: for every $k \in \mathbb{Z}$,
$$\gcd\big(f(k),\, g(k)\big) \ \big|\ \operatorname{Res}(f,g),$$
since evaluating $uf+vg = \operatorname{Res}(f,g)$ at $x=k$ exhibits the resultant as a
$\mathbb{Z}$-combination of $f(k)$ and $g(k)$. The converse fails, and the reason is instructive: a
prime with $p \mid \operatorname{Res}(f,g)$ gives a common factor of $\bar f,\bar g$ in
$\mathbb{F}_p[x]$, but that factor need not have degree $1$, so there need be no *rational point*
$k$ where both vanish. For $f = x^3+2x+5$, $g = x^2-3$ and $p = 5$ the common factor is $t^2+2$,
irreducible over $\mathbb{F}_5$: no such $k$ exists, though $5 \mid \operatorname{Res}(f,g) = -50$.

In [7]:
# Remark 5.4: the lcm restatement fires when one leading coefficient dies
f, g, p = x, 2*x + 1, 2
m, n = f.degree(), g.degree()
S.<t> = GF(p)[]
fb, gb = S(f.list()), S(g.list())
print(f"f = {f}, g = {g},  Res = {f.resultant(g)}")
print(f"  mod {p}:  fbar = {fb} (deg {fb.degree()}),  gbar = {gb} (deg {gb.degree()})")
print(f"  gcd = {gcd(fb, gb)},  deg lcm = {lcm(fb, gb).degree()} < m+n = {m+n}")
print(f"  lcm criterion fires, yet {p} does not divide Res = {f.resultant(g)}")

# Remark 5.5: gcd of values always divides the resultant, but not conversely
f, g = x^3 + 2*x + 5, x^2 - 3
r = f.resultant(g)
print(f"\nRes = {r}")
print("  gcd(f(k),g(k)), k = -6..6:", [gcd(f(k), g(k)) for k in range(-6, 7)])
print("  every one divides Res:", all(r % gcd(f(k), g(k)) == 0 for k in range(-50, 51)))
S5.<t> = GF(5)[]
d = gcd(S5(f.list()), S5(g.list()))
print(f"  gcd mod 5 = {d}, irreducible over F_5: {d.is_irreducible()}")
print("  k in F_5 with 5 | f(k) and 5 | g(k):", [k for k in range(5) if f(k) % 5 == 0 and g(k) % 5 == 0])

f = x, g = 2*x + 1,  Res = 1
  mod 2:  fbar = t (deg 1),  gbar = 1 (deg 0)
  gcd = 1,  deg lcm = 1 < m+n = 2
  lcm criterion fires, yet 2 does not divide Res = 1

Res = -50
  gcd(f(k),g(k)), k = -6..6: [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1]
  every one divides Res: True
  gcd mod 5 = t^2 + 2, irreducible over F_5: True
  k in F_5 with 5 | f(k) and 5 | g(k): []


### 7.1 The contraction $I = (f,g)\cap\mathbb{Z}$

Remark 5.1a suggests the right home for the criterion: not the gcd, but the ideal
$I = (f,g)\cap\mathbb{Z}$ of $\mathbb{Z}$ — the *contraction*, or the elimination ideal of the pair.
Since $\mathbb{Z}$ is a PID, $I = (d)$ for a unique $d \ge 0$.

**Proposition 6.** *Let $f,g \in \mathbb{Z}[x]$ have degrees $m,n\ge1$ and leading coefficients
$a_m,b_n$, let $p$ be prime, and let $I = (f,g)\cap\mathbb{Z}$. Then*

* **(a)** $\big(\operatorname{Res}(f,g)\big) \subseteq I$;
* **(b)** $I \subseteq (p) \implies p \mid \operatorname{Res}(f,g)$, with no hypotheses;
* **(c)** *if $p \nmid a_m$ or $p \nmid b_n$, then*
  $$p \mid \operatorname{Res}(f,g) \iff I \subseteq (p) \iff \gcd(\bar f,\bar g)\neq 1 .$$

*Proof.* (a) is Proposition 1: $\operatorname{Res}(f,g) = uf+vg$ lies in $(f,g)$, and it is an integer,
so it lies in $(f,g)\cap\mathbb{Z} = I$; being an ideal of $\mathbb{Z}$, $I$ then contains the whole
ideal generated by it. (b) is immediate from (a): $I \subseteq (p)$ forces
$\operatorname{Res}(f,g) \in (p)$.

(c) Only one new ingredient is needed, and it is two lines:

> **Lemma A.** *If $\gcd(\bar f,\bar g) \ne 1$ in $\mathbb{F}_p[x]$ then $I \subseteq (p)$.*
>
> *Proof.* Let $m \in I$, say $m = af+bg$ with $a,b\in\mathbb{Z}[x]$. Reducing mod $p$,
> $\bar m = \bar a\bar f + \bar b\bar g$ is divisible by the common factor $\gcd(\bar f,\bar g)$,
> which has positive degree. But $\bar m$ is a constant, and a nonconstant polynomial divides a
> constant only if that constant is $0$. So $\bar m = 0$, i.e. $p \mid m$. $\square$

Now close the cycle: $I \subseteq (p) \Rightarrow p \mid \operatorname{Res}$ is (b);
$p\mid\operatorname{Res} \Rightarrow \gcd(\bar f,\bar g)\ne1$ is Theorem 5, using the hypothesis on the
leading coefficients; and $\gcd(\bar f,\bar g)\ne1 \Rightarrow I\subseteq(p)$ is Lemma A. $\blacksquare$

**Remark 6.1 (both containments are strict in general).** In (a), $f = x^2$, $g = x^2+p$ give
$I = (p)$ while $\operatorname{Res}(f,g) = p^2$: the resultant records the intersection with
multiplicity, so it lies in $I$ without generating it. What (c) says is that $I$ and
$(\operatorname{Res}(f,g))$ nevertheless have the same prime divisors — away from the leading-coefficient
case, $\sqrt{I} = \sqrt{(\operatorname{Res}(f,g))}$.

**Remark 6.2 (the hypothesis in (c) cannot be dropped).** For $f = 2x^2+1$, $g = 2x^2+x+1$ and $p=2$
we have $\operatorname{Res}(f,g) = 2$, so $2 \mid \operatorname{Res}$; but $g - f = x$ and then
$f - x\cdot 2x = 1$, so $(f,g) = \mathbb{Z}[x]$ and $I = (1) \not\subseteq (2)$. Both leading
coefficients are even, so (c) does not apply — and note (b) survives, as it must: $(2) \subseteq (1)$.

In [8]:
M = PolynomialRing(ZZ, 'x', 1)     # ideal arithmetic over ZZ needs the multivariate constructor

def contraction(f, g):
    "the generator d >= 0 of I = (f,g) cap ZZ, read off a Groebner basis over ZZ"
    G = M.ideal(M(str(f)), M(str(g))).groebner_basis()
    consts = [ZZ(h) for h in G if h.degree() <= 0]
    return gcd(consts) if consts else ZZ(0)

for f, g in [(x, x+7), (x^2, x^2+7), (x^3+2*x+5, x^2-3), (2*x^2+1, 2*x^2+x+1)]:
    d, r = contraction(f, g), f.resultant(g)
    print(f"f={str(f):12s} g={str(g):12s}  I=({d})   Res={r:5d}   "
          f"(Res) <= I: {d != 0 and r % d == 0}   I = (Res): {d == abs(r)}")

f=x            g=x + 7         I=(7)   Res=    7   (Res) <= I: True   I = (Res): True
f=x^2          g=x^2 + 7       I=(7)   Res=   49   (Res) <= I: True   I = (Res): False
f=x^3 + 2*x + 5 g=x^2 - 3       I=(10)   Res=  -50   (Res) <= I: True   I = (Res): False
f=2*x^2 + 1    g=2*x^2 + x + 1  I=(1)   Res=    2   (Res) <= I: True   I = (Res): False


In [9]:
# Proposition 6, checked: (b) unconditionally, and (c) whenever the caveat does not apply
set_random_seed(7)
def data(f, g, p):
    d   = contraction(f, g)
    S   = GF(p)['t']
    return (d,
            f.resultant(g) % p == 0,                       # (i)   p | Res
            gcd(S(f.list()), S(g.list())).degree() > 0,    # (ii)  gcd != 1 mod p
            d == 0 or d % p == 0,                          # (iii) I <= (p)
            f.leading_coefficient() % p == 0 and g.leading_coefficient() % p == 0)

n, bad_b, bad_c = 0, 0, 0
for _ in range(80):
    f = R.random_element(degree=(1,4), x=-6, y=6)
    g = R.random_element(degree=(1,4), x=-6, y=6)
    if f.degree() < 1 or g.degree() < 1: continue
    for p in prime_range(30):
        d, i, ii, iii, caveat = data(f, g, p); n += 1
        if iii and not i:                       bad_b += 1     # (b) must hold always
        if not caveat and not (i == ii == iii): bad_c += 1     # (c) off the caveat
print(f"{n} instances:  violations of (b): {bad_b}    violations of (c): {bad_c}")

# and the caveat really is what breaks it
brk = sum(1 for _ in range(80)
          for f, g in [(R(2*ZZ.random_element(-3,4)*x^2 + ZZ.random_element(-5,6)*x + ZZ.random_element(-5,6)),
                        R(2*ZZ.random_element(-3,4)*x^2 + ZZ.random_element(-5,6)*x + ZZ.random_element(-5,6)))]
          if f.degree() >= 1 and g.degree() >= 1 and data(f, g, 2)[1] != data(f, g, 2)[3])
print("pairs with both leading coefficients even where p|Res and I<=(p) disagree:", brk)

800 instances:  violations of (b): 0    violations of (c): 0
pairs with both leading coefficients even where p|Res and I<=(p) disagree: 40


**Corollary 6.3 (the whole story as one chain).** *If $\gcd(a_m,b_n) = 1$ — in particular if either
$f$ or $g$ is monic — then*
$$\big(\operatorname{Res}(f,g)\big) \;\subseteq\; I \;\subseteq\; \sqrt{\big(\operatorname{Res}(f,g)\big)},$$
*and consequently $\sqrt{I} = \sqrt{(\operatorname{Res}(f,g))}$.*

*Proof.* The left inclusion is Proposition 6(a), which needs no hypothesis. For the right one, write
$I = (d)$ and $\sqrt{(\operatorname{Res})} = (r)$ with $r$ the squarefree kernel of
$\operatorname{Res}(f,g)$; then $I \subseteq \sqrt{(\operatorname{Res})}$ says exactly that
$p \mid \operatorname{Res}(f,g) \Rightarrow p \mid d$, i.e. $p \mid \operatorname{Res} \Rightarrow I \subseteq (p)$
for every $p$ — which is Proposition 6(c), applicable at every prime because no prime divides both
leading coefficients. Taking radicals in the chain gives
$\sqrt{(\operatorname{Res})} \subseteq \sqrt{I} \subseteq \sqrt{(\operatorname{Res})}$. $\blacksquare$

The chain is *equivalent* to Proposition 6: the left inclusion encodes $I \subseteq (p) \Rightarrow p \mid \operatorname{Res}$
and the right one the converse, for all $p$ at once. Read it as: **the resultant is a computable element
of the elimination ideal (left), and it is sharp up to multiplicities (right)** — it detects no spurious
primes. The two inclusions are generally strict, and $I$ is the finer invariant: for $f = x^3+2x+5$,
$g = x^2-3$ we get $(50) \subsetneq (10) = I \subseteq (10)$.

Degenerate case: if $\operatorname{Res}(f,g) = 0$ the chain reads $(0) \subseteq I \subseteq (0)$, and
indeed $I = 0$ — a vanishing resultant means a common factor of positive degree in $\mathbb{Q}[x]$
(Theorem 2), and no nonzero integer is divisible by such a factor.

Without the hypothesis the right inclusion fails, by Remark 6.2: there $I = (1)$ while
$\sqrt{(\operatorname{Res})} = (2)$.

In [10]:
def rad(n): return ZZ(0) if n == 0 else prod(p for p, _ in factor(n))

def chain(f, g):
    "does (Res) <= I <= rad(Res) hold?"
    d, r = contraction(f, g), f.resultant(g)
    left  = (d != 0 and r % d == 0) or (r == 0 and d == 0)
    right = (d == 0) if rad(r) == 0 else (d != 0 and d % rad(r) == 0)
    return d, r, left, right

for f, g in [(x, x+7), (x^2, x^2+7), (x^3+2*x+5, x^2-3), (2*x^2+x, 3*x^2+1),
             (x*(2*x+1), (x+1)*(2*x+1)), (2*x^2+1, 2*x^2+x+1)]:
    d, r, l, ri = chain(f, g)
    print(f"({r:5d}) <= ({d:3d}) <= ({rad(r):3d}):  left {str(l):5s} right {str(ri):5s}"
          f"   lc's coprime: {gcd(f.leading_coefficient(), g.leading_coefficient()) == 1}"
          f"   f={f}, g={g}")

set_random_seed(3)
tally = {True: [0, 0], False: [0, 0]}      # keyed by "leading coefficients coprime"
for _ in range(150):
    f = R.random_element(degree=(1,4), x=-6, y=6)
    g = R.random_element(degree=(1,4), x=-6, y=6)
    if f.degree() < 1 or g.degree() < 1: continue
    cop = gcd(f.leading_coefficient(), g.leading_coefficient()) == 1
    d, r, l, ri = chain(f, g)
    tally[cop][0] += 1
    tally[cop][1] += (l and ri)
for cop in (True, False):
    tot, good = tally[cop]
    print(f"\nlc's coprime = {cop}:  chain holds in {good}/{tot} pairs")

(    7) <= (  7) <= (  7):  left True  right True    lc's coprime: True   f=x, g=x + 7
(   49) <= (  7) <= (  7):  left True  right True    lc's coprime: True   f=x^2, g=x^2 + 7
(  -50) <= ( 10) <= ( 10):  left True  right True    lc's coprime: True   f=x^3 + 2*x + 5, g=x^2 - 3
(    7) <= (  7) <= (  7):  left True  right True    lc's coprime: True   f=2*x^2 + x, g=3*x^2 + 1
(    0) <= (  0) <= (  0):  left True  right True    lc's coprime: False   f=2*x^2 + x, g=2*x^2 + 3*x + 1
(    2) <= (  1) <= (  2):  left True  right False   lc's coprime: False   f=2*x^2 + 1, g=2*x^2 + x + 1

lc's coprime = True:  chain holds in 99/99 pairs

lc's coprime = False:  chain holds in 18/51 pairs


In [11]:
# Machine check of Theorem 5 over many pairs and all p < 300
set_random_seed(1)
def bad_by_gcd(f, g, p):
    S = GF(p)['t']
    return gcd(S(f.list()), S(g.list())).degree() > 0
def bad_by_lc(f, g, p):
    return f.leading_coefficient() % p == 0 and g.leading_coefficient() % p == 0

trials, ok = 0, True
for _ in range(60):
    f = R.random_element(degree=(1,4), x=-8, y=8)
    g = R.random_element(degree=(1,4), x=-8, y=8)
    if f.degree() < 1 or g.degree() < 1: continue
    r = f.resultant(g)
    if r == 0: continue
    for p in prime_range(300):
        trials += 1
        lhs = (r % p == 0)
        rhs = bad_by_gcd(f, g, p) or bad_by_lc(f, g, p)
        if lhs != rhs:
            ok = False; print("COUNTEREXAMPLE:", f, g, p)
print(f"Theorem 5 verified in {trials} instances:", ok)

Theorem 5 verified in 3720 instances: True


## 8. What it is good for

**8.1 Discriminants and ramification.** For $\deg f = m$ the discriminant is
$$\operatorname{disc}(f) = \frac{(-1)^{m(m-1)/2}}{a_m}\operatorname{Res}(f,f').$$
Since $f$ has a repeated root iff $\gcd(f,f') \ne 1$, Theorem 5 says: for $p \nmid a_m$,
$$p \mid \operatorname{disc}(f) \iff \bar f \text{ has a repeated factor in } \mathbb{F}_p[x].$$
For $f$ monic irreducible with root $\theta$, the resultant computes the discriminant of the
*monogenic order*: $\operatorname{disc}(\mathbb{Z}[\theta]) = \operatorname{disc}(f)$. The link with
ramification is then the index formula
$$\operatorname{disc}(f) = \big[\mathcal{O}_K : \mathbb{Z}[\theta]\big]^2 \cdot \operatorname{disc}(K),$$
so that, since $p$ ramifies in $K$ exactly when $p \mid \operatorname{disc}(K)$, the primes dividing
$\operatorname{Res}(f,f')$ are the ramified primes **together with** the primes dividing the index.
That is why one can only say $\mathbb{Z}[\theta]$ *can* ramify there. Dedekind's own example shows the
gap is real: for $f = x^3-x^2-2x-8$ one has $\operatorname{disc}(f) = -2012 = -2^2\cdot 503$ while
$\operatorname{disc}(K) = -503$ and $[\mathcal{O}_K:\mathbb{Z}[\theta]] = 2$ — the field is not
monogenic, and $2$ divides the resultant for that reason alone, not because of ramification.
Separating the two cases is Dedekind's criterion, itself a computation with $\bar f \in \mathbb{F}_p[x]$.

**8.2 Elimination.** For $f,g \in K[x,y]$, the resultant taken with respect to $y$ lies in $K[x]$
and vanishes exactly at those $x$ where the two curves have a common $y$ — that is, it *eliminates*
$y$. This gives implicit equations of rational curves: parametrise the nodal cubic by
$x = t^2-1$, $y = t^3-t$, then eliminating $t$ from $t^2-1-x$ and $t^3-t-y$ recovers $y^2 = x^3+x^2$.
(Theorem 5 is the arithmetic analogue: there we eliminate nothing, but specialise $\mathbb{Z}\to\mathbb{F}_p$.)

**8.3 Arithmetic of algebraic numbers.** If $\alpha$ is a root of $f$ and $\beta$ of $g$, then
$\operatorname{Res}_y\!\big(f(y),\, g(x-y)\big)$ is a polynomial in $x$ killing $\alpha+\beta$ — by
Proposition 4 its roots are exactly the $\alpha_i + \beta_j$. Likewise
$\operatorname{Res}_y(f(y), y^{\deg g}g(x/y))$ handles products. This is how one proves that algebraic
numbers form a field without invoking any structure theory.

In [12]:
# 8.1 discriminant and repeated factors mod p
h = x^3 - x - 1
print("disc =", h.discriminant(), " = (-1)^3 * Res(h,h') =", (-1)^3 * h.resultant(h.derivative()))
for p in [5, 23]:
    S.<t> = GF(p)[]
    print(f"  mod {p:2d}: {factor(S(h.list()))}   repeated factor: {not S(h.list()).is_squarefree()}")

disc = -23  = (-1)^3 * Res(h,h') = -23
  mod  5: (t + 3) * (t^2 + 2*t + 3)   repeated factor: False
  mod 23: (t + 20) * (t + 13)^2   repeated factor: True


In [13]:
# 8.2 implicitisation of the nodal cubic
P.<t, X, Y> = QQ[]
print("Res_t(t^2-1-X, t^3-t-Y) =", (t^2 - 1 - X).resultant(t^3 - t - Y, t))

# 8.3 minimal polynomial of sqrt(2) + sqrt(3)
Q.<y, z> = QQ[]
res = (y^2 - 2).resultant((z - y)^2 - 3, y)
print("Res_y(y^2-2, (z-y)^2-3) =", res.factor())
print("check:", QQbar(sqrt(2) + sqrt(3)).minpoly())

Res_t(t^2-1-X, t^3-t-Y) = -X^3 - X^2 + Y^2
Res_y(y^2-2, (z-y)^2-3) = z^4 - 10*z^2 + 1
check: x^4 - 10*x^2 + 1


**8.4 Norms are resultants.** This is Proposition 4 read in a number field, and it is the sharpest
arithmetic use of the whole theory.

> **Proposition 7.** *Let $f$ be the monic minimal polynomial of $\theta$, $K = \mathbb{Q}(\theta)$ of
> degree $m$, and $g \in \mathbb{Q}[x]$. Then*
> $$N_{K/\mathbb{Q}}\big(g(\theta)\big) = \operatorname{Res}(f,g).$$
>
> *Proof.* By Proposition 4 with $a_m = 1$, $\operatorname{Res}(f,g) = \prod_i g(\alpha_i)$ over the
> roots of $f$; the $\alpha_i$ are the conjugates of $\theta$, so the $g(\alpha_i)$ are the conjugates
> of $g(\theta)$, whose product is the norm. $\square$

Two consequences. Taking $g = a - bx$ gives the **norm form**
$$N_{K/\mathbb{Q}}(a - b\theta) = \operatorname{Res}(f, a-bx) = b^{m} f(a/b),$$
the homogenised minimal polynomial — the computation at the heart of the number field sieve, where one
needs norms of $a - b\theta$ for millions of pairs $(a,b)$ and gets them by evaluating one binary form.
And if $f,g$ are *monic* with integer coefficients, then $\operatorname{Res}_y(f(y), g(x-y))$ is monic
in $x$ with integer coefficients, which is the resultant proof that the algebraic **integers** form a
ring — the refinement of §8.3 from field to ring.

**8.5 Cyclotomic polynomials.** A classical evaluation, which we verify below:
$$\big|\operatorname{Res}(\Phi_m,\Phi_n)\big| = \begin{cases} p^{\varphi(n)} & \text{if } m/n \text{ is a power of a prime } p,\\ 1 & \text{otherwise,}\end{cases} \qquad (m > n \ge 1).$$
Fed into Theorem 5 this is very sharp. When $m/n$ is not a prime power the resultant is a unit, so
**$\Phi_m$ and $\Phi_n$ stay coprime modulo every prime**; when $m = n p^k$ they acquire a common
factor exactly modulo $p$ and no other prime. Specialising to $n=1$ recovers $\Phi_m(1) = p$ if
$m = p^k$ and $1$ otherwise, hence $N(1-\zeta_p) = p$ — the standard route to $p$ being totally
ramified in $\mathbb{Q}(\zeta_{p^k})$ and unramified elsewhere.

**8.6 Bézout's theorem.** For forms $F,G \in K[X,Y,Z]$ of degrees $m,n$ with no common component,
$\operatorname{Res}_Z(F,G)$ is a form in $X,Y$ of degree exactly $mn$ (each entry of the Sylvester
matrix is homogeneous, and the determinant is homogeneous of degree $mn$). Its $mn$ roots in
$\mathbb{P}^1$ are the projections from $(0{:}0{:}1)$ of the intersection points, so two plane curves of
degrees $m$ and $n$ meet in exactly $mn$ points counted with multiplicity. Elimination *is* the classical
proof of Bézout.

**8.7 How resultants are actually computed.** Never by expanding an $(m+n)\times(m+n)$ determinant. From
$\operatorname{Res}(g,f) = \operatorname{lc}(g)^{\deg f - \deg r}\operatorname{Res}(g,r)$ for
$r = f \bmod g$ — immediate from the product formula, since $f$ and $r$ agree at the roots of $g$ — one
gets the Euclidean recursion
$$\operatorname{Res}(f,g) = (-1)^{mn}\operatorname{lc}(g)^{\,m-\deg r}\operatorname{Res}(g,\, f \bmod g),$$
with base case $\operatorname{Res}(f,c) = c^{m}$ for constant $c$. That is polynomial time. Over
$\mathbb{Z}$ the intermediate coefficients still blow up, which is what the *subresultant* PRS (or a
modular/CRT method, using Lemma 5 to reduce mod many primes) is for.

**8.8 Elliptic curves.** The discriminant of a Weierstrass model is a resultant, so primes of bad
reduction are read off the same way; resultants of division polynomials detect shared torsion, and
resultants of modular polynomials are used in isogeny computations and in the SEA point-counting
algorithm.

In [14]:
# 8.1 (refined): disc(f) = [O_K : Z[theta]]^2 * disc(K)
RQ.<t> = QQ[]
for poly in [t^3 - t - 1, t^3 - t^2 - 2*t - 8, t^3 - 2]:
    K.<a> = NumberField(poly)
    df, dK = poly.discriminant(), K.discriminant()
    idx = K.order(a).index_in(K.maximal_order())
    print(f"f = {poly}")
    print(f"   disc(Z[theta]) = disc(f) = {df} = {factor(df)},   disc(K) = {dK},   index = {idx}")
    print(f"   index^2 * disc(K) == disc(f): {idx^2*dK == df}")
    print(f"   ramified: {[p for p,_ in factor(dK)]}      dividing disc(f): {[p for p,_ in factor(df)]}")

# 8.4 norms as resultants
print("\n--- norms ---")
f = t^3 - t - 1
K.<th> = NumberField(f)
for g in [t^2 + 1, 2*t - 3, t^2 - t + 5]:
    print(f"   N({g}) = {g(th).norm()} = Res(f,g) = {f.resultant(g)}")
for a, b in [(3, 2), (5, -1), (7, 4)]:
    print(f"   N({a} - {b}*theta) = {(a - b*th).norm()},  Res(f, {a}-{b}x) = {f.resultant(a - b*t)},"
          f"  b^3 f(a/b) = {b^3 * f(QQ(a)/QQ(b))}")

f = t^3 - t - 1
   disc(Z[theta]) = disc(f) = -23 = -1 * 23,   disc(K) = -23,   index = 1
   index^2 * disc(K) == disc(f): True
   ramified: [23]      dividing disc(f): [23]
f = t^3 - t^2 - 2*t - 8
   disc(Z[theta]) = disc(f) = -2012 = -1 * 2^2 * 503,   disc(K) = -503,   index = 2
   index^2 * disc(K) == disc(f): True
   ramified: [503]      dividing disc(f): [2, 503]
f = t^3 - 2
   disc(Z[theta]) = disc(f) = -108 = -1 * 2^2 * 3^3,   disc(K) = -108,   index = 1
   index^2 * disc(K) == disc(f): True
   ramified: [2, 3]      dividing disc(f): [2, 3]

--- norms ---
   N(t^2 + 1) = 5 = Res(f,g) = 5
   N(2*t - 3) = -7 = Res(f,g) = -7
   N(t^2 - t + 5) = 191 = Res(f,g) = 191
   N(3 - 2*theta) = 7,  Res(f, 3-2x) = 7,  b^3 f(a/b) = 7
   N(5 - -1*theta) = 121,  Res(f, 5--1x) = 121,  b^3 f(a/b) = 121
   N(7 - 4*theta) = 167,  Res(f, 7-4x) = 167,  b^3 f(a/b) = 167


In [15]:
# 8.5 resultants of cyclotomic polynomials
print("m, n with |Res(Phi_m, Phi_n)| != 1   (m <= 12):")
for m in range(2, 13):
    for n in range(1, m):
        r = cyclotomic_polynomial(m).resultant(cyclotomic_polynomial(n))
        if abs(r) != 1:
            q = ZZ(m/n) if (QQ(m)/QQ(n)).denominator() == 1 else None
            p = q.prime_factors()[0] if (q and q.is_prime_power()) else None
            print(f"   Res(Phi_{m:2d}, Phi_{n:2d}) = {r:5d}    m/n = {QQ(m)/QQ(n)}"
                  + (f"   = {p}^phi({n}) = {p^euler_phi(n)}" if p else "   (not a prime power!)"))
print("\nand when m/n is NOT a prime power the resultant is a unit, so by Theorem 5")
print("Phi_m and Phi_n stay coprime mod every prime, e.g. m=12, n=5:",
      cyclotomic_polynomial(12).resultant(cyclotomic_polynomial(5)))

# 8.6 Bezout
P.<X, Y, Z> = QQ[]
F = X^3 + Y^3 + Z^3 - 3*X*Y*Z
G = X^2 + Y^2 - 2*Z^2
res = F.resultant(G, Z)
print(f"\nBezout: deg F = {F.degree()}, deg G = {G.degree()},"
      f" Res_Z(F,G) homogeneous of degree {res.degree()} = {F.degree()*G.degree()},"
      f" homogeneous: {res.is_homogeneous()}")

# 8.7 the Euclidean recursion
f = 3*t^4 - t^3 + 2*t - 5
g = 2*t^3 + t^2 - 7
m, n = f.degree(), g.degree()
r = f % g
lhs = f.resultant(g)
rhs = (-1)^(m*n) * g.leading_coefficient()^(m - r.degree()) * g.resultant(r)
print(f"\nEuclid: Res(f,g) = {lhs},  (-1)^(mn) lc(g)^(m-deg r) Res(g, f mod g) = {rhs},  equal: {lhs == rhs}")

m, n with |Res(Phi_m, Phi_n)| != 1   (m <= 12):
   Res(Phi_ 2, Phi_ 1) =    -2    m/n = 2   = 2^phi(1) = 2
   Res(Phi_ 3, Phi_ 1) =     3    m/n = 3   = 3^phi(1) = 3
   Res(Phi_ 4, Phi_ 1) =     2    m/n = 4   = 2^phi(1) = 2
   Res(Phi_ 4, Phi_ 2) =     2    m/n = 2   = 2^phi(2) = 2
   Res(Phi_ 5, Phi_ 1) =     5    m/n = 5   = 5^phi(1) = 5
   Res(Phi_ 6, Phi_ 2) =     3    m/n = 3   = 3^phi(2) = 3
   Res(Phi_ 6, Phi_ 3) =     4    m/n = 2   = 2^phi(3) = 4
   Res(Phi_ 7, Phi_ 1) =     7    m/n = 7   = 7^phi(1) = 7
   Res(Phi_ 8, Phi_ 1) =     2    m/n = 8   = 2^phi(1) = 2
   Res(Phi_ 8, Phi_ 2) =     2    m/n = 4   = 2^phi(2) = 2
   Res(Phi_ 8, Phi_ 4) =     4    m/n = 2   = 2^phi(4) = 4
   Res(Phi_ 9, Phi_ 1) =     3    m/n = 9   = 3^phi(1) = 3
   Res(Phi_ 9, Phi_ 3) =     9    m/n = 3   = 3^phi(3) = 9
   Res(Phi_10, Phi_ 2) =     5    m/n = 5   = 5^phi(2) = 5
   Res(Phi_10, Phi_ 5) =    16    m/n = 2   = 2^phi(5) = 16
   Res(Phi_11, Phi_ 1) =    11    m/n = 11   = 11^phi(1) = 11
   R

**8.9 Weil reciprocity, and what the sign $(-1)^{mn}$ really is.** On a smooth projective curve $X$
over an algebraically closed field, Weil reciprocity states that for $f,g \in k(X)^\times$
$$\prod_{P \in X} (f,g)_P = 1, \qquad (f,g)_P = (-1)^{\operatorname{ord}_P(f)\operatorname{ord}_P(g)}\left[\frac{f^{\operatorname{ord}_P(g)}}{g^{\operatorname{ord}_P(f)}}\right](P),$$
the *tame symbol* at $P$. For $X = \mathbb{P}^1$ this is not merely provable by resultants — it **is**
Proposition 4.

Let $f,g \in k[x]$ be monic and coprime, of degrees $m,n$, with roots $\alpha_i,\beta_j$. Then:

* at a root $\alpha$ of $f$: $\operatorname{ord}(f) = 1$, $\operatorname{ord}(g) = 0$, so $(f,g)_\alpha = g(\alpha)^{-1}$;
* at a root $\beta$ of $g$: $(f,g)_\beta = f(\beta)$;
* at infinity: $\operatorname{ord}_\infty(f) = -m$, $\operatorname{ord}_\infty(g) = -n$, so
  $$(f,g)_\infty = (-1)^{mn}\left[\frac{g^{m}}{f^{n}}\right](\infty) = (-1)^{mn},$$
  because $g^m/f^n$ has degree $mn - mn = 0$ and leading coefficient $1$.

Multiplying and setting the product to $1$ gives
$\prod_i g(\alpha_i) = (-1)^{mn}\prod_j f(\beta_j)$, that is,
$$\operatorname{Res}(f,g) = (-1)^{mn}\operatorname{Res}(g,f),$$
the symmetry in §5. So **the sign $(-1)^{mn}$ is the tame symbol at infinity.** It looks like
bookkeeping in the determinant picture, and it is a geometric contribution: two polynomials both have
poles at $\infty$, so their divisors are never disjoint there, and the naive form
$\prod_i g(\alpha_i) = \prod_j f(\beta_j)$ genuinely fails whenever $mn$ is odd.

For a general curve the picture is more honest as follows. Resultants give the case $X=\mathbb{P}^1$
outright, and that is the base of the standard proof; the reduction of general $X$ to $\mathbb{P}^1$ goes
by pushforward along a nonconstant $\pi : X \to \mathbb{P}^1$, using that the tame symbol commutes with
the norm $N_{k(X)/k(\mathbb{P}^1)}$ — a separate ingredient, not itself a resultant identity. What
resultants *do* supply is the norm: by §8.4, for a plane curve $F(x,y) = 0$ with $F$ monic in $y$,
$$N_{k(X)/k(x)}(g) = \operatorname{Res}_y(F,g),$$
so the pushforward in that proof is an explicit resultant computation. Taking $F = y^2 - x^3 - ax - b$
and $g = y - (\lambda x + \mu)$ returns the line function of the chord–tangent law — the function
Miller's algorithm evaluates, in a setting where Weil reciprocity is exactly what makes the Weil
pairing well defined.

In [16]:
# 8.9  Weil reciprocity on P^1 is the resultant symmetry; the sign lives at infinity
RQ.<u> = QQ[]
print("f, g monic and coprime:   prod g(alpha) = (-1)^(mn) * prod f(beta)\n")
for f, g in [(u^3 - u - 1, u - 2), (u^3 - u - 1, u^2 + 3*u + 1), (u^2 + 1, u - 3)]:
    m, n = f.degree(), g.degree()
    Pf = QQ(prod(g(a) for a in f.roots(QQbar, multiplicities=False)))   # = Res(f,g)
    Pg = QQ(prod(f(b) for b in g.roots(QQbar, multiplicities=False)))   # = Res(g,f)
    sign = (-1)^(m*n)                                                   # = (f,g)_infinity
    print(f"  f = {str(f):16s} g = {str(g):14s} mn = {m*n}")
    print(f"     prod g(alpha) = {Pf},  prod f(beta) = {Pg},  (f,g)_oo = {sign}")
    print(f"     reciprocity: {Pf == sign*Pg}      naive version without the sign: {Pf == Pg}")

# the full tame-symbol product over all of P^1, infinity included
f, g = u^3 - u - 1, u - 2
sym  = prod(g(a)^(-1) for a in f.roots(QQbar, multiplicities=False))   # at the zeros of f
sym *= prod(f(b)      for b in g.roots(QQbar, multiplicities=False))   # at the zeros of g
sym *= (-1)^(f.degree()*g.degree())                                    # at infinity
print(f"\n  product of all tame symbols = {QQ(sym)}   (Weil reciprocity)")

# the norm/pushforward as a resultant (cf. 8.4), on an elliptic curve
S.<X, Y> = QQ[]
F = Y^2 - X^3 - 2*X - 3
print()
for g2 in [Y - (3*X + 1), Y - 5, X - 7]:
    print(f"  N_(k(C)/k(x))({g2}) = Res_Y(F, g) = {F.resultant(g2, Y)}")
print("  the first is the line function of the chord-tangent law")

f, g monic and coprime:   prod g(alpha) = (-1)^(mn) * prod f(beta)

  f = u^3 - u - 1      g = u - 2          mn = 3
     prod g(alpha) = -5,  prod f(beta) = 5,  (f,g)_oo = -1
     reciprocity: True      naive version without the sign: False
  f = u^3 - u - 1      g = u^2 + 3*u + 1  mn = 6
     prod g(alpha) = 11,  prod f(beta) = 11,  (f,g)_oo = 1
     reciprocity: True      naive version without the sign: True
  f = u^2 + 1          g = u - 3          mn = 2
     prod g(alpha) = 10,  prod f(beta) = 10,  (f,g)_oo = 1
     reciprocity: True      naive version without the sign: True

  product of all tame symbols = 1   (Weil reciprocity)

  N_(k(C)/k(x))(-3*X + Y - 1) = Res_Y(F, g) = -X^3 + 9*X^2 + 4*X - 2
  N_(k(C)/k(x))(Y - 5) = Res_Y(F, g) = -X^3 - 2*X + 22
  N_(k(C)/k(x))(X - 7) = Res_Y(F, g) = X^2 - 14*X + 49
  the first is the line function of the chord-tangent law


## 9. Bad reduction, and other closed conditions on $\operatorname{Spec}\mathbb{Z}$

Everything so far was one variable. The questions that matter for a variety over $\mathbb{Z}$ are
multivariate, and they need generalisations of the resultant — which do exist, and are exactly the
right tool.

### 9.1 The principle

Let $X \subseteq \mathbb{P}^n_{\mathbb{Z}}$ be cut out by forms $f_1,\dots,f_r \in \mathbb{Z}[x_0,\dots,x_n]$,
and let $B \subseteq X$ be a locus defined by a *closed* condition (singularity, tangency, rank drop, …).
Then:

1. $B$ is closed in $X$, and $X \to \operatorname{Spec}\mathbb{Z}$ is **proper**;
2. so the image of $B$ in $\operatorname{Spec}\mathbb{Z}$ is closed — the fundamental theorem of
   elimination theory;
3. a closed proper subset of $\operatorname{Spec}\mathbb{Z}$ is a **finite set of primes**, cut out by an
   ideal $I \subseteq \mathbb{Z}$ — the elimination ideal, exactly as in §7.1;
4. an *eliminant* (a resultant) is an explicit element of $I$ with the same radical.

That last step is §7.1 verbatim: $(\operatorname{Res}) \subseteq I \subseteq \sqrt{(\operatorname{Res})}$.
The resultant is the computable proxy; the contraction $I$ is the sharp invariant. The whole of §7 is the
case $n = 1$, $B$ = "the two points collide".

### 9.2 Which generalisations

* **Macaulay resultant.** For $n+1$ forms in $n+1$ variables, a single integer polynomial in their
  coefficients vanishing exactly when they have a common zero in $\mathbb{P}^n$. This is *the* direct
  generalisation of Theorem 2, and Sage implements it as `macaulay_resultant`.
* **Discriminant of a form.** $\operatorname{Disc}(F) = \operatorname{Res}(\partial_0F,\dots,\partial_nF)$
  up to a factor which is a power of $\deg F$ — the hypersurface analogue of §8.1.
* **Sparse (mixed, GKZ) resultants.** Adapted to prescribed Newton polytopes; much smaller than the
  Macaulay resultant when the systems are sparse.
* **Fitting and determinantal ideals.** Minors of a matrix, for rank conditions.
* **Chow forms**, for a variety rather than a complete intersection.
* **Gröbner elimination over $\mathbb{Z}$**, which is what one actually runs — as in §7.1.

### 9.3 Bad reduction of a hypersurface

For $X = V(F) \subseteq \mathbb{P}^n_{\mathbb{Z}}$, the fibre $X_p$ is singular exactly when $F$ and all its
partials acquire a common zero mod $p$, so
$$\{\text{primes of bad reduction of this model}\} = \{p : p \mid \operatorname{Disc}(F)\}.$$
Two caveats, both real:

* **The Euler relation degenerates at $p \mid \deg F$.** Since $d\cdot F = \sum_i x_i\,\partial_i F$, the
  partials alone cut out the singular locus only when $p \nmid d$; for $p \mid d$ one must keep $F$ in the
  system. This is the source of the constant factor relating $\operatorname{Res}(\partial F)$ to
  $\operatorname{Disc}(F)$, and it means the raw resultant of the partials can report primes dividing $d$
  that are perfectly good. For the elliptic curve below, $\operatorname{Res}(\partial F) = 3^3\cdot|\Delta|$
  and the curve is smooth mod $3$.
* **The eliminant sees the *model*, not the variety.** More on this in §9.5.

### 9.4 General $X$, and a trap

Beyond hypersurfaces, take $J$ = the ideal generated by the $f_i$ together with the maximal minors of the
Jacobian, and compute $I = J \cap \mathbb{Z}$. The trap: done naively this always gives $I = 0$. Every
generator is homogeneous of positive degree, so every $\mathbb{Z}$-combination of them has zero constant
term — the origin of the affine cone is a "singular point" over every prime. One must first saturate with
respect to the irrelevant ideal $(x_0,\dots,x_n)$, i.e. ask that the *projective* scheme be empty. The
Macaulay resultant has this built in.

In [17]:
# 9.3 bad reduction of a plane cubic, and the spurious factor from Euler
P.<X, Y, Z> = QQ[]
F = Y^2*Z - X^3 - 2*X*Z^2 - 3*Z^3
resP = P.macaulay_resultant([F.derivative(v) for v in (X, Y, Z)])
Delta = EllipticCurve([0, 0, 0, 2, 3]).discriminant()
print("Res(partials) =", factor(resP))
print("Delta         =", factor(Delta))
print("ratio         =", resP / abs(Delta), "= 3^3, a power of deg F = 3")
print("is the curve smooth mod 3?", Delta % 3 != 0, " -> the prime 3 is spurious")
print("true bad primes:", [p for p, _ in factor(EllipticCurve([0,0,0,2,3]).conductor())])

Res(partials) = 2^4 * 3^3 * 5^2 * 11
Delta         = -1 * 2^4 * 5^2 * 11
ratio         = 27 = 3^3, a power of deg F = 3
is the curve smooth mod 3? True  -> the prime 3 is spurious
true bad primes: [2, 5, 11]


### 9.5 Model versus intrinsic — the theme of the whole note

The eliminant measures the degeneracies of the *presentation*. Three instances, all the same shape:

| setting | eliminant of the presentation | intrinsic invariant | discrepancy |
|---|---|---|---|
| §7.1 | $\operatorname{Res}(f,g)$ | $I = (f,g)\cap\mathbb{Z}$ | multiplicities |
| §8.1 | $\operatorname{disc}(f) = \operatorname{disc}(\mathbb{Z}[\theta])$ | $\operatorname{disc}(K)$ | $[\mathcal{O}_K:\mathbb{Z}[\theta]]^2$ |
| §9.3 | $\Delta$ of a Weierstrass model | minimal $\Delta$, the conductor | $u^{12}$ |

In each row the presentation overshoots, and the correction is a perfect power. The leading-coefficient
caveat of Theorem 5 is the same phenomenon in its simplest form: the resultant notices that the degree
dropped, which is a fact about how the polynomial was written, not about its roots.

### 9.6 Other closed conditions, and some non-examples

Closed on $\operatorname{Spec}\mathbb{Z}$, hence a finite set of primes, and detected by an eliminant:

* **a system acquiring a common zero** — Macaulay resultant; the direct generalisation of Theorem 5;
* **fibres becoming singular** — discriminant (§9.3);
* **a finite flat map becoming ramified** — discriminant of the order (§8.1); the branch locus;
* **rank drop of a matrix**, or degeneracy of a map of vector bundles — determinantal / Fitting ideals,
  i.e. the ideal of $k\times k$ minors;
* **two subvarieties meeting, or meeting non-transversally** — resultant of the pair; §7 is the case of
  two points on $\mathbb{P}^1$;
* **jumping of fibre dimension** — upper semicontinuous, so the jump locus is closed;
* **loss of geometric integrality of the fibres** — geometric integrality is open for proper flat families,
  so its failure is closed (though the eliminant is much less explicit here).

Not of this kind, worth keeping straight: the Galois group of $\bar f$ dropping (by Chebotarev these primes
have positive density — an infinite, non-closed set), the rank of a Mordell–Weil group, or the existence of
an $\mathbb{F}_p$-point. Those are not closed conditions and no resultant will detect them.

In [18]:
# 9.5 a non-minimal model reports a prime that is not bad
E = EllipticCurve('37a')
E2 = E.change_weierstrass_model(1/2, 0, 0, 0)
print("minimal model :", E, "  Delta =", factor(E.discriminant()))
print("rescaled model:", E2, "  Delta =", factor(E2.discriminant()))
print("   Delta scaled by u^12:", E2.discriminant() == 2^12 * E.discriminant())
print("   primes dividing Delta of the rescaled model:", [p for p, _ in factor(E2.discriminant())])
print("   true bad primes (conductor)               :", [p for p, _ in factor(E2.conductor())])

minimal model : Elliptic Curve defined by y^2 + y = x^3 - x over Rational Field   Delta = 37
rescaled model: Elliptic Curve defined by y^2 + 8*y = x^3 - 16*x over Rational Field   Delta = 2^12 * 37
   Delta scaled by u^12: True
   primes dividing Delta of the rescaled model: [2, 37]
   true bad primes (conductor)               : [37]


In [19]:
# 9.6 two closed conditions, measured
print("--- rank drop of an integer matrix: the primes dividing the maximal minors ---")
A = matrix(ZZ, [[2,3,5], [7,11,13], [1,4,9]])
print("   det =", A.det(), "=", factor(A.det()))
for p in [2, 3, 5, 7, 11, 13, 29, 31]:
    print(f"     rank mod {p:2d} = {A.change_ring(GF(p)).rank()}    p divides det: {A.det() % p == 0}")

print("\n--- three conics in P^2: a common point exactly at the primes dividing the Macaulay resultant ---")
C = [X^2 + Y^2 - Z^2, X*Y - Z^2 + X*Z, Y^2 + X*Z - 2*Y*Z]
res = P.macaulay_resultant(C)
print("   Macaulay resultant =", res, "=", factor(res))
for p in prime_range(45):
    Fp = GF(p)
    pts = [pt for pt in ProjectiveSpace(Fp, 2).rational_points()
           if all(c.change_ring(Fp)(*pt) == 0 for c in C)]
    if res % p == 0 or pts:
        print(f"     p = {p}:  p divides Res: {res % p == 0}    common F_p-points: {pts}")

--- rank drop of an integer matrix: the primes dividing the maximal minors ---
   det = 29 = 29
     rank mod  2 = 3    p divides det: False
     rank mod  3 = 3    p divides det: False
     rank mod  5 = 3    p divides det: False
     rank mod  7 = 3    p divides det: False
     rank mod 11 = 3    p divides det: False
     rank mod 13 = 3    p divides det: False
     rank mod 29 = 2    p divides det: True
     rank mod 31 = 3    p divides det: False

--- three conics in P^2: a common point exactly at the primes dividing the Macaulay resultant ---
   Macaulay resultant = -13 = -1 * 13
     p = 13:  p divides Res: True    common F_p-points: [(2 : 6 : 1)]


## Summary

| statement | why |
|---|---|
| $\operatorname{Res}_{m,n}(f,g) = \det\Phi$, $\Phi(u,v) = uf+vg$ | definition, read correctly (§2) |
| $uf + vg = \operatorname{Res}(f,g)$ for some $u,v$ | adjugate of the Sylvester matrix (§3) |
| over a field: $\operatorname{Res} = 0 \iff$ common factor | $\det\Phi = 0 \iff \Phi$ non-injective (§4) |
| $\operatorname{Res}_{m,n}$ commutes with any $\varphi: R \to S$ | $\det$ of a matrix of coefficients (§6) |
| $p \mid \operatorname{Res}(f,g) \iff \bar f,\bar g$ share a factor mod $p$ | the previous two, over $\mathbb{F}_p$ (§7) |

The one thing to carry away is the role of **formal degrees**: they are what makes base change
unconditional, and the price is the leading-coefficient clause in Theorem 5 — the case where a
common root escapes to infinity mod $p$.

**Further reading.** Lang, *Algebra*, IV §8; Cox–Little–O'Shea, *Ideals, Varieties and Algorithms*,
ch. 3 (elimination); Gelfand–Kapranov–Zelevinsky, *Discriminants, Resultants and Multidimensional
Determinants* for the modern view.